# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadTalha-pk/ML-Internship-Task1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Baseline rule: I will prioritize pages that are still visible but have not been updated recently. A page is considered stale when it has not been updated for at least 91 days, and visible when it has at least 500 impressions in the trailing 90-day window. Among these candidates, pages with more impressions will rank higher because they have more exposed traffic to protect. The reason code will be stale_visible_page, and the action will be review_for_refresh.
Before using the rule, I will check two signals: staleness and impression volume. trend_direction will be used only to measure whether the signals are associated with the starter decline label; it will not be used in the baseline score.

In [5]:
import os
import subprocess
import pandas as pd
import numpy as np

REPO_DIR = "/content/ML_Internship_FlyRankAI"

if not os.path.exists(REPO_DIR):
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/MuhammadTalha-pk/ML_Internship_FlyRankAI",
            REPO_DIR
        ],
        check=True
    )

os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Evaluation label only — NEVER a baseline feature
df["is_declining_check"] = (
    df["trend_direction"].str.lower().eq("down").astype(int)
)

# -----------------------------
# Signal 1: staleness
# Use 100+ impressions to reduce low-volume noise
# -----------------------------
stale_check = df[df["impressions_90d"] >= 100].copy()

stale_check["staleness_bucket"] = pd.cut(
    stale_check["days_since_last_update"],
    bins=[-1, 30, 90, 180, np.inf],
    labels=["0-30", "31-90", "91-180", "181+"]
)

staleness_table = (
    stale_check
    .groupby("staleness_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        decline_rate=("is_declining_check", "mean")
    )
    .reset_index()
)

staleness_table["decline_rate"] = (
    staleness_table["decline_rate"].round(3)
)

print("SIGNAL 1 — STALENESS")
display(staleness_table)


# -----------------------------
# Signal 2: impression volume
# -----------------------------
df["volume_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[0, 100, 500, 3000, 30000, np.inf],
    labels=[
        "1-99",
        "100-499",
        "500-2,999",
        "3,000-29,999",
        "30,000+"
    ],
    right=False
)

volume_table = (
    df
    .groupby("volume_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        decline_rate=("is_declining_check", "mean")
    )
    .reset_index()
)

volume_table["decline_rate"] = (
    volume_table["decline_rate"].round(3)
)

print("\nSIGNAL 2 — IMPRESSION VOLUME")
display(volume_table)

SIGNAL 1 — STALENESS


,staleness_bucket,n,decline_rate
0,0-30,13735,0.583
1,31-90,152,0.592
2,91-180,8084,0.622
3,181+,35,0.743



SIGNAL 2 — IMPRESSION VOLUME


,volume_bucket,n,decline_rate
0,1-99,7994,0.389
1,100-499,5280,0.604
2,"500-2,999",8443,0.621
3,"3,000-29,999",7205,0.586
4,"30,000+",1078,0.462


Staleness verdict: CONFIRMED. Among pages with at least 100 impressions, the observed decline rate generally rises as the time since the last update increases. The oldest bucket is small, so I will treat the pattern as directional rather than causal.
Volume verdict: MIXED. Decline rates are higher in several middle-volume groups, but they do not increase consistently with impression volume. Therefore, I will use impressions mainly to prioritize the potential impact of already-stale pages, not as evidence that high-volume pages are automatically more likely to decline.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Baseline scoring rule: A page receives a positive baseline score only when it is both stale (days_since_last_update >= 91) and visible (impressions_90d >= 500). Eligible pages are ranked by their trailing 90-day impressions. The rule uses only information available before any outcome label and does not use trend_direction or trend_pct as inputs.

In [7]:
import os
import json
import numpy as np

# Transparent rule components
stale = (df["days_since_last_update"] >= 91).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)

# One simple baseline score
df["baseline_score"] = stale * visible * df["impressions_90d"]

# Exactly one reason code and one action label
df["reason_code"] = np.where(
    (stale == 1) & (visible == 1),
    "stale_visible_page",
    "not_flagged"
)

df["action_label"] = np.where(
    df["baseline_score"] > 0,
    "review_for_refresh",
    "no_priority_action"
)

# Rank highest-score pages first
queue = df.sort_values(
    ["baseline_score", "impressions_90d"],
    ascending=False
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

# Evaluation label ONLY — never used to calculate baseline_score
labels = queue["is_declining_check"].to_numpy()

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = df["is_declining_check"].mean()
p50 = precision_at_k(
    df["baseline_score"],
    df["is_declining_check"],
    50
)

print("Base decline rate:", round(base_rate, 3))
print("Baseline Precision@50:", round(p50, 3))
print("Flagged pages:", int((df["baseline_score"] > 0).sum()))

# Write required CSV
os.makedirs("work/outputs", exist_ok=True)

queue_output = queue[
    [
        "rank",
        "content_id",
        "client_id",
        "baseline_score",
        "impressions_90d",
        "days_since_last_update",
        "avg_position",
        "ctr",
        "reason_code",
        "action_label"
    ]
]

csv_path = "work/outputs/baseline_action_score.csv"
queue_output.to_csv(csv_path, index=False)

# Small metrics receipt that CAN be committed
metrics = {
    "base_decline_rate": round(float(base_rate), 4),
    "baseline_precision_at_50": round(float(p50), 4),
    "flagged_pages": int((df["baseline_score"] > 0).sum())
}

with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Wrote:", csv_path)
print("Wrote: work/outputs/baseline_metrics.json")

queue_output.head(10)

Base decline rate: 0.542
Baseline Precision@50: 0.44
Flagged pages: 6575
Wrote: work/outputs/baseline_action_score.csv
Wrote: work/outputs/baseline_metrics.json


,rank,content_id,client_id,baseline_score,impressions_90d,days_since_last_update,avg_position,ctr,reason_code,action_label
0,1,content_5fe46e04994d,client_4e07408562,517715,517715,104,4.2,0.14,stale_visible_page,review_for_refresh
1,2,content_2dba2b1f9536,client_6208ef0f77,443434,443434,104,27.9,0.21,stale_visible_page,review_for_refresh
2,3,content_2c2606c5d176,client_19581e27de,347399,347399,104,4.2,0.53,stale_visible_page,review_for_refresh
3,4,content_cb112fce36be,client_19581e27de,309910,309910,104,5.6,0.16,stale_visible_page,review_for_refresh
4,5,content_9532f197bbc8,client_4e07408562,309192,309192,104,2.0,0.87,stale_visible_page,review_for_refresh
5,6,content_36ff89c8214e,client_19581e27de,295097,295097,104,7.3,0.05,stale_visible_page,review_for_refresh
6,7,content_b28d1efd668f,client_6208ef0f77,286608,286608,104,26.2,0.06,stale_visible_page,review_for_refresh
7,8,content_813e88069237,client_6208ef0f77,233561,233561,104,26.2,0.06,stale_visible_page,review_for_refresh
8,9,content_c21024970297,client_19581e27de,211366,211366,104,5.1,0.41,stale_visible_page,review_for_refresh
9,10,content_c8e9d6ab9013,client_19581e27de,208678,208678,104,9.7,0.00,stale_visible_page,review_for_refresh


Baseline result: The overall decline rate is 54.2%, while the baseline Precision@50 is 44.0%. This means the current rule does not outperform the overall base rate for identifying declining pages. I will keep this baseline frozen because its purpose is to provide a transparent comparison point for the later model. The rule may still be useful for prioritizing stale pages with high exposure, but it should not be treated as a strong predictor of decline.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:
top10_review = queue_output.head(10).copy()

def review_reason(row):
    return (
        f"Stale for {int(row['days_since_last_update'])} days "
        f"with {int(row['impressions_90d']):,} impressions; "
        "ranked highly because exposure is large."
    )

def confidence_note(row):
    pos = row["avg_position"]

    if 0 < pos <= 10:
        return "Medium-high: strong visibility, but freshness may not be the real problem."
    elif 10 < pos <= 20:
        return "Medium: visible page, but ranking position may also matter."
    else:
        return "Medium-low: high exposure exists, but weaker position may have causes unrelated to freshness."

def wrong_reason(row):
    pos = row["avg_position"]
    ctr = row["ctr"]

    if 0 < pos <= 3:
        return "The page may already perform strongly; refreshing it could be unnecessary."
    elif ctr == 0:
        return "The real issue may be snippet/intent mismatch rather than content freshness."
    elif pos > 20:
        return "Weak position may come from relevance, competition, or demand rather than staleness."
    else:
        return "Seasonality, consolidation, or another factor could explain performance instead of freshness."

top10_review["why_it_is_here"] = top10_review.apply(
    review_reason, axis=1
)

top10_review["confidence_note"] = top10_review.apply(
    confidence_note, axis=1
)

top10_review["what_would_make_it_wrong"] = top10_review.apply(
    wrong_reason, axis=1
)

review_columns = [
    "rank",
    "content_id",
    "action_label",
    "reason_code",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "why_it_is_here",
    "confidence_note",
    "what_would_make_it_wrong"
]

top10_review[review_columns]

,rank,content_id,action_label,reason_code,impressions_90d,days_since_last_update,avg_position,ctr,why_it_is_here,confidence_note,what_would_make_it_wrong
0,1,content_5fe46e04994d,review_for_refresh,stale_visible_page,517715,104,4.2,0.14,"Stale for 104 days with 517,715 impressions; r...","Medium-high: strong visibility, but freshness ...","Seasonality, consolidation, or another factor ..."
1,2,content_2dba2b1f9536,review_for_refresh,stale_visible_page,443434,104,27.9,0.21,"Stale for 104 days with 443,434 impressions; r...","Medium-low: high exposure exists, but weaker p...","Weak position may come from relevance, competi..."
2,3,content_2c2606c5d176,review_for_refresh,stale_visible_page,347399,104,4.2,0.53,"Stale for 104 days with 347,399 impressions; r...","Medium-high: strong visibility, but freshness ...","Seasonality, consolidation, or another factor ..."
3,4,content_cb112fce36be,review_for_refresh,stale_visible_page,309910,104,5.6,0.16,"Stale for 104 days with 309,910 impressions; r...","Medium-high: strong visibility, but freshness ...","Seasonality, consolidation, or another factor ..."
4,5,content_9532f197bbc8,review_for_refresh,stale_visible_page,309192,104,2.0,0.87,"Stale for 104 days with 309,192 impressions; r...","Medium-high: strong visibility, but freshness ...",The page may already perform strongly; refresh...
5,6,content_36ff89c8214e,review_for_refresh,stale_visible_page,295097,104,7.3,0.05,"Stale for 104 days with 295,097 impressions; r...","Medium-high: strong visibility, but freshness ...","Seasonality, consolidation, or another factor ..."
6,7,content_b28d1efd668f,review_for_refresh,stale_visible_page,286608,104,26.2,0.06,"Stale for 104 days with 286,608 impressions; r...","Medium-low: high exposure exists, but weaker p...","Weak position may come from relevance, competi..."
7,8,content_813e88069237,review_for_refresh,stale_visible_page,233561,104,26.2,0.06,"Stale for 104 days with 233,561 impressions; r...","Medium-low: high exposure exists, but weaker p...","Weak position may come from relevance, competi..."
8,9,content_c21024970297,review_for_refresh,stale_visible_page,211366,104,5.1,0.41,"Stale for 104 days with 211,366 impressions; r...","Medium-high: strong visibility, but freshness ...","Seasonality, consolidation, or another factor ..."
9,10,content_c8e9d6ab9013,review_for_refresh,stale_visible_page,208678,104,9.7,0.00,"Stale for 104 days with 208,678 impressions; r...","Medium-high: strong visibility, but freshness ...",The real issue may be snippet/intent mismatch ...


Top-10 review: All ten pages were selected because they satisfy the same transparent rule: they are stale, still visible, and have very high impression exposure. The manual review also shows why the rule is imperfect. Some pages already rank strongly, some have weaker positions that may reflect relevance or competition rather than freshness, and a high-exposure page does not automatically need a refresh. Human review is therefore still required before taking action.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: The top-ranked queue contains several cases where the rule may be wrong. Rank 5 already has a very strong average position of 2.0, so age alone may not justify a refresh. Ranks 2, 7, and 8 have average positions around 26–28, where weak relevance or competition may matter more than freshness. Rank 10 has a page-one average position but zero CTR, which may indicate a click-through or intent problem rather than a freshness problem. These cases show why the baseline is decision-support, not an automatic refresh instruction.

In [9]:
# Leakage check: only these two fields calculate the baseline score
baseline_inputs = [
    "days_since_last_update",
    "impressions_90d"
]

forbidden_fields = [
    "trend_direction",
    "trend_pct",
    "is_declining_check"
]

leaked_fields = [
    col for col in forbidden_fields
    if col in baseline_inputs
]

print("Baseline inputs:", baseline_inputs)
print("Forbidden/label-derived fields used:", leaked_fields)
print("Leakage check passed:", len(leaked_fields) == 0)

Baseline inputs: ['days_since_last_update', 'impressions_90d']
Forbidden/label-derived fields used: []
Leakage check passed: True


Leakage check: The baseline score uses only days_since_last_update and impressions_90d. trend_direction, trend_pct, and the evaluation label are excluded from the rule and are used only after scoring to evaluate performance. No future-window or label-derived input is used to rank the pages.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.